# Regresion polinomica

La regresion polinomica extiende la regresion lineal agregando potencias de los predictores (`x`, `x^2`, `x^3`, ...) como nuevas features. Sigue siendo **lineal en los coeficientes**, asi que se ajusta con OLS, pero modela relaciones no lineales.

Usamos datos sinteticos con una relacion cubica + ruido para ver underfitting, buen ajuste y overfitting segun el grado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(42)
n = 80
X = np.sort(rng.uniform(-3, 3, n)).reshape(-1, 1)
y_true = 0.5 * X.ravel() ** 3 - X.ravel() ** 2 + 2
y = y_true + rng.normal(0, 3, n)
X.shape, y.shape

In [ ]:
plt.scatter(X, y, s=15, color="black", alpha=0.6)
plt.xlabel("x"); plt.ylabel("y"); plt.title("Datos sinteticos (relacion cubica + ruido)");

## Ajuste por grado

`PolynomialFeatures(d)` genera las potencias hasta grado `d`; lo encadenamos con `LinearRegression` en un `Pipeline`. Comparamos grado 1 (recta, underfit), grado 3 (el real) y grado 15 (overfit).

In [ ]:
def fit_poly(degree):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X, y)
    return model

xx = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)

plt.scatter(X, y, s=15, color="black", alpha=0.4)
for d in [1, 3, 15]:
    plt.plot(xx, fit_poly(d).predict(xx), label=f"grado {d}")
plt.ylim(y.min() - 3, y.max() + 3)
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.title("Ajuste segun el grado");

## Curva de error vs grado (bias-variance)

Partimos en train/test y medimos RMSE en ambos para cada grado. El error de train baja siempre; el de test baja y luego sube: ahi empieza el sobreajuste.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

degrees = range(1, 16)
train_rmse, test_rmse = [], []
for d in degrees:
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    model.fit(X_train, y_train)
    train_rmse.append(np.sqrt(mean_squared_error(y_train, model.predict(X_train))))
    test_rmse.append(np.sqrt(mean_squared_error(y_test, model.predict(X_test))))

mejor = list(degrees)[int(np.argmin(test_rmse))]
print("Grado con menor RMSE de test:", mejor)

plt.plot(list(degrees), train_rmse, marker="o", label="train")
plt.plot(list(degrees), test_rmse, marker="o", label="test")
plt.axvline(mejor, color="gray", linestyle="--", alpha=0.6)
plt.xlabel("grado del polinomio"); plt.ylabel("RMSE"); plt.legend();

El grado que minimiza el RMSE de **test** es el mejor compromiso: grados bajos hacen underfitting (mucho sesgo) y grados altos overfitting (mucha varianza, se ajustan al ruido).

El optimo exacto depende del split concreto (con este conjunto de test chico cae cerca del grado real de los datos, 3, pero puede variar). Por eso la **validacion cruzada** da una estimacion mas estable del mejor grado que un unico train/test split.